# Cell Tracking Submission
This notebook generates and validates the competition submission file.

Attach the competition test data and the tracking source before running all cells. The final file is written to `/kaggle/working/submission.csv`.

In [1]:
from pathlib import Path
import sys

KAGGLE_ROOT = Path('/kaggle/input')
RUNNING_ON_KAGGLE = KAGGLE_ROOT.exists()
OUTPUT_PATH = Path('/kaggle/working/submission.csv') if RUNNING_ON_KAGGLE else Path.cwd() / 'submission.csv'
source_roots = [Path('/kaggle/working'), KAGGLE_ROOT, Path.cwd()] if RUNNING_ON_KAGGLE else [Path.cwd()]

# Find tracking source whether it is at the root or inside an attached Kaggle dataset.
for root in list(source_roots):
    if root.exists():
        source_roots.extend(path.parent for path in root.rglob('tracking') if path.is_dir())
source_roots = list(dict.fromkeys(source_roots))
for root in source_roots:
    if (root / 'tracking').is_dir() and str(root) not in sys.path:
        sys.path.insert(0, str(root))

label_stores = sorted(
    {
        path
        for root in source_roots
        if root.exists()
        for path in root.rglob('*')
        if (path.is_dir() and path.name.endswith('.zarr'))
        or (path.is_file() and (path.name.endswith('_labels.npz') or path.name.endswith('_labels.npy')))
    },
    key=str,
)
if not label_stores:
    raise FileNotFoundError('No .zarr, *_labels.npz, or *_labels.npy test stores were found.')

print('Discovered label stores:')
for path in label_stores:
    print(path)
print(f'Output: {OUTPUT_PATH}')

Discovered label stores:
/Users/manmeet/Desktop/Bio Hub Project/kagglehub/kalman_labels.npz
/Users/manmeet/Desktop/Bio Hub Project/kagglehub/test2_labels.npz
/Users/manmeet/Desktop/Bio Hub Project/kagglehub/test_labels.npz
/Users/manmeet/Desktop/Bio Hub Project/kagglehub/train_labels.npz
Output: /Users/manmeet/Desktop/Bio Hub Project/kagglehub/submission.csv


In [2]:
from tracking.pipeline import run_directory

# The parent directory is used because run_directory discovers each dataset store by name.
# For a Kaggle dataset containing nested stores, point TEST_ROOT at that dataset folder.
TEST_ROOT = label_stores[0].parent
run_directory(
    str(TEST_ROOT),
    str(OUTPUT_PATH),
    link=True,
    max_dist_um=7.0,
    voxel_size=(1.625, 0.40625, 0.40625),
)
print(f'Created {OUTPUT_PATH}')

Prepared kalman: 3 nodes
Prepared test2: 2 nodes
Prepared test: 2 nodes
Prepared train: 2 nodes
Wrote /Users/manmeet/Desktop/Bio Hub Project/kagglehub/submission.csv with 9 nodes and 5 edges
Created /Users/manmeet/Desktop/Bio Hub Project/kagglehub/submission.csv


In [ ]:
import csv

required = {'id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id'}
with OUTPUT_PATH.open(newline='') as handle:
    rows = list(csv.DictReader(handle))

if not rows:
    raise ValueError('Submission is empty.')
if set(rows[0]) != required:
    raise ValueError(f'Unexpected columns: {set(rows[0])}')
if [row['id'] for row in rows] != [str(index) for index in range(len(rows))]:
    raise ValueError('The id column must be consecutive.')
node_ids = {row['node_id'] for row in rows if row['row_type'] == 'node'}
for row in rows:
    if not row['dataset']:
        raise ValueError('Every row must include a dataset name.')
    if row['row_type'] == 'node':
        for field in ('t', 'z', 'y', 'x'):
            if not float(row[field]).is_integer():
                raise ValueError(f'Node coordinate {field} is not an integer.')
    elif row['row_type'] == 'edge':
        if row['source_id'] not in node_ids or row['target_id'] not in node_ids:
            raise ValueError('Edge references an unknown node.')
    else:
        raise ValueError(f'Unsupported row type: {row["row_type"]}')

print(f'Validated {len(rows)} rows across {len({row["dataset"] for row in rows})} datasets.')
print(f'Upload this file: {OUTPUT_PATH}')